# YZM212 Makine Öğrenmesi - 2. Laboratuvar Ödevi
## MLE ile Akıllı Şehir Planlaması: Trafik Yoğunluğu Modellemesi

Bu not defterinde, bir caddeden 1 dakikada geçen araç sayılarına ilişkin veriler kullanılarak Poisson Dağılımı'nın λ (lambda) parametresi Maximum Likelihood Estimation (MLE) yöntemiyle tahmin edilecektir. İlk olarak sayısal optimizasyon yöntemiyle MLE bulunacak, ardından modelin veriye uyumu görselleştirilecek ve son olarak aykırı değerlerin (outlier) MLE üzerindeki etkisi analiz edilecektir.

In [ ]:
# Gerekli kütüphaneleri içe aktaralım
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
from scipy.special import factorial  # faktöriyel hesaplamak için (opsiyonel)

# Grafiklerin notebook içinde görünmesi için sihirli komut
%matplotlib inline

## Bölüm 2: Python ile Sayısal (Numerical) MLE

### 1. Verinin Yüklenmesi

In [ ]:
# Gözlemlenen Trafik Verisi (1 dakikada geçen araç sayısı)
traffic_data = np.array([12, 15, 10, 8, 14, 11, 13, 16, 9, 12, 11, 14, 10, 15])

# Veriyi hızlıca inceleyelim
print("Veri seti:", traffic_data)
print("Veri setinin boyutu (n):", len(traffic_data))
print("Veri setinin ortalaması:", np.mean(traffic_data))

### 2. Negatif Log-Olabilirlik (Negative Log-Likelihood) Fonksiyonunun Yazılması

Teorik türetmede Log-Likelihood fonksiyonunu $\ell(\lambda) = -n\lambda + (\sum k_i) \ln(\lambda) - \sum \ln(k_i!)$ olarak bulmuştuk. Optimizasyon sırasında $\sum \ln(k_i!)$ terimi $\lambda$'ya bağlı olmadığı için sabittir ve minimizasyonu etkilemez. Bu nedenle bu terimi ihmal edebiliriz. 

Bizim amacımız *Negatif* Log-Likelihood'ı minimize etmek (yani Likelihood'ı maksimize etmek) olduğu için, fonksiyonumuzu şu şekilde yazacağız:
$$ \text{NLL}(\lambda) = -\ell(\lambda) \approx n\lambda - (\sum k_i) \ln(\lambda) $$

In [ ]:
def negative_log_likelihood(lam, data):
    """
    Poisson dağılımı için Negatif Log-Likelihood (NLL) hesaplar.
    Optimizasyon sırasında sabit olan log(k!) terimleri ihmal edilmiştir.
    
    Parametreler:
    lam (float): Poisson dağılımının lambda parametresi (ortalama).
    data (numpy array): Gözlem verileri.
    
    Return:
    float: Hesaplanan NLL değeri.
    """
    n = len(data)
    sum_k = np.sum(data)
    
    # Negatif Log-likelihood (sabit terimler hariç)
    # nll = n*lam - sum_k * log(lam)   şeklinde olmalı.
    # Ancak optimizasyon fonksiyonları genelde minimizasyon yaptığı için doğrudan bu formülü kullanıyoruz.
    # Küçük bir sayısal kararlılık için lambda'nın çok küçük olduğu durumlarda log(lam) -sonsuz olmasın diye
    # bir önlem alınabilir ama biz optimize ederken alt sınır koyacağız.
    nll = n * lam - sum_k * np.log(lam)
    
    return nll

### 3. Fonksiyonun Minimizasyonu (Sayısal MLE)

In [ ]:
# Başlangıç tahmini: lambda için rastgele bir pozitif sayı seçelim
initial_guess = 1.0

# Optimizasyon: Negatif Log-Likelihood'ı minimize et
# lambda > 0 olması gerektiği için bounds kullanıyoruz. (0.001, None) ile 0'dan büyük olmasını sağlıyoruz.
result = opt.minimize(negative_log_likelihood, 
                       initial_guess, 
                       args=(traffic_data,), 
                       bounds=[(0.001, None)])

# Sonuçları yazdıralım
print("********** MLE SONUÇLARI **********")
print(f"Sayısal Optimizasyon ile Bulunan MLE Lambda: {result.x[0]:.4f}")
print(f"Analitik Çözüm (Verilerin Ortalaması): {np.mean(traffic_data):.4f}")
print(f"Optimizasyon başarılı mı? {result.success}")
print(f"Optimizasyon mesajı: {result.message}")
print("***********************************")

# Bulunan lambda değerini bir değişkene atayalım
mle_lambda = result.x[0]

## Bölüm 3: Model Karşılaştırma ve Görselleştirme

### Poisson PMF Grafiği ve Veri Histogramı

Şimdi bulduğumuz $\lambda \approx 12.0$ değerini kullanarak teorik Poisson dağılımını çizeceğiz ve bunu gerçek verilerimizin histogramıyla karşılaştıracağız.

In [ ]:
# Histogram için gerekli aralıkları belirleyelim. Veri 8 ile 16 arasında.
bins = np.arange(traffic_data.min() - 0.5, traffic_data.max() + 1.5, 1) # 7.5'ten 16.5'e 1'er adım

# 1. Gerçek verilerin histogramını çizelim (density=True ile olasılık yoğunluğuna çeviriyoruz)
#    Böylece Poisson PMF ile karşılaştırılabilir oluyor.
plt.hist(traffic_data, bins=bins, density=True, alpha=0.7, color='black', edgecolor='black', label='Gerçek Veri (Histogram)')

# 2. Poisson PMF'ini hesaplayalım ve çizelim
k_values = np.arange(traffic_data.min(), traffic_data.max() + 1) # Grafikte göstereceğimiz k değerleri
# Poisson PMF formülü: P(k) = (lambda^k * e^{-lambda}) / k!
pmf_values = (mle_lambda**k_values * np.exp(-mle_lambda)) / factorial(k_values)

plt.plot(k_values, pmf_values, 'ro-', markersize=8, linewidth=2, label=f'Poisson PMF (λ={mle_lambda:.2f})')

# Grafik düzenlemeleri
plt.xlabel('Bir Dakikada Geçen Araç Sayısı (k)')
plt.ylabel('Olasılık')
plt.title('Model Uyumu: Gerçek Veri ve Poisson Dağılımı')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.xticks(np.arange(8, 18, 1)) # x eksenindeki sayıları netleştirelim

# Grafiği göster
plt.show()

### Görsel Yorum
Yukarıdaki grafikte, siyah çubuklar gerçek verilerimizin dağılımını, kırmızı çizgi ve noktalar ise MLE ile bulduğumuz Poisson modelinin tahmin ettiği olasılıkları göstermektedir. Görüldüğü gibi Poisson eğrisi, veri setinin tepe noktasını (yaklaşık 10-13 arası) ve genel şeklini oldukça iyi yakalamıştır. Bu görsel uyum, trafik verisinin Poisson dağılımı ile modellenebileceği varsayımımızı desteklemektedir.

## Bölüm 4: Gerçek Hayat Senaryosu - "Outlier" Analizi

### MLE'nin Aykırı Değerlere Karşı Hassasiyeti

Şimdi veri setimize yanlışlıkla kaydedilmiş, gerçek dışı büyük bir değer (outlier) ekleyelim ve MLE tahmininin nasıl etkilendiğini gözlemleyelim.

In [ ]:
# 1. Orijinal veri setine 200 değerini outlier olarak ekleyelim
traffic_data_with_outlier = np.append(traffic_data, 200)

print("=== AYKIRI DEĞER (OUTLIER) ANALİZİ ===")
print(f"Orijinal veri seti boyutu: {len(traffic_data)}, Ortalaması: {np.mean(traffic_data):.2f}")
print(f"Yeni veri seti boyutu (outlier eklenmiş): {len(traffic_data_with_outlier)}")

# 2. Yeni veri seti için MLE'yi (yani ortalamayı) hesaplayalım
new_mean = np.mean(traffic_data_with_outlier)
print(f"\n--> Outlier eklendikten sonraki ortalama (yeni MLE λ'sı): {new_mean:.2f}")

# Etkiyi daha net görmek için optimizasyonu da yapalım (aynı sonucu verecektir)
result_outlier = opt.minimize(negative_log_likelihood, 
                               initial_guess, 
                               args=(traffic_data_with_outlier,), 
                               bounds=[(0.001, None)])
print(f"Sayısal MLE ile bulunan yeni lambda: {result_outlier.x[0]:.2f}")

print("\n--- GÖRSELLEŞTİRME: Outlier'ın Etkisi ---")
# Grafik ile de gösterelim (isteğe bağlı)
plt.figure(figsize=(12,4))

# Alt grafik 1: Orijinal dağılım
plt.subplot(1,2,1)
plt.hist(traffic_data, bins=np.arange(5,20,1), alpha=0.7, color='green', edgecolor='black')
plt.axvline(np.mean(traffic_data), color='red', linestyle='dashed', linewidth=2, label=f'Ortalama={np.mean(traffic_data):.2f}')
plt.title('Orijinal Veri Dağılımı')
plt.xlabel('Araç Sayısı')
plt.ylabel('Frekans')
plt.legend()

# Alt grafik 2: Outlier'lı dağılım
plt.subplot(1,2,2)
plt.hist(traffic_data_with_outlier, bins=np.arange(5,210,10), alpha=0.7, color='orange', edgecolor='black')
plt.axvline(new_mean, color='red', linestyle='dashed', linewidth=2, label=f'Ortalama={new_mean:.2f}')
plt.title('Outlier Eklenmiş Veri Dağılımı')
plt.xlabel('Araç Sayısı')
plt.ylabel('Frekans')
plt.legend()

plt.tight_layout()
plt.show()

### Sonuç ve Tartışma
Görüldüğü gibi, tek bir aykırı değer (200), ortalama tahminimizi yaklaşık **12'den 24.5'e** çıkararak ciddi şekilde saptırmıştır. Bu, MLE yönteminin (ve dolayısıyla ortalamanın) aykırı değerlere karşı ne kadar hassas olduğunu açıkça göstermektedir.

Eğer bir belediye bu hatalı veriyle yol genişletme veya sinyalizasyon sürelerini ayarlama kararı alsaydı, gereksiz yere büyük bütçeli projelere girişebilir ve şehrin trafik dengesini bozabilirdi. Bu nedenle, veri bilimi projelerinde veri ön işleme (aykırı değer tespiti ve temizliği) aşaması, model kurmak kadar önemlidir. Veri setimizde böyle bir aykırı değer varsa, ya bu değer düzeltilmeli ya da ortanca (median) gibi aykırı değerlerden daha az etkilenen (robust) yöntemler kullanılmalıdır.